# Final Submission Guidance: Comparing Models, Tuning, and Features

*Checkpoint 1 Guidance* prepared one data set and cross-validated one model. This notebook covers what the final submission adds: trying several model families, tuning one setting, and engineering features, with cross-validation evidence for every choice. It uses the same Adult census data and the same pipeline as *Checkpoint 1 Guidance*, so none of its code is a Spaceship Titanic answer, but almost all of it is code you'll adapt.

Like *Checkpoint 1 Guidance*, this notebook is a reference: Each of its first four sections correspond to Sections 2–5 of `final.ipynb`. You're encouraged to go back and forth between this guide and `final.ipynb`, piece by piece for help with syntax, and occasionally, a new concept you need to complete certain tasks.

| This notebook | `final.ipynb` |
| --- | --- |
| [1 · A menu of model families](#section-1) | 2 · Compare model families |
| [2 · Comparing families on the same folds](#section-2) | 2 · Compare model families |
| [3 · Tuning one setting](#section-3) | 3 · Tune one setting |
| [4 · Engineering features](#section-4) | 4 · Engineer a feature |
| [5 · Optional: the wrong way to cross-validate](#section-5) | (not graded) |

### By the end, you'll 

1. Build a pipeline for any of five model families, and name one or two settings of each that are most worth changing or tuning.
2. Compare models fairly: the same folds for every model, in one table of mean ± standard deviation.
3. Tune one setting with cross-validation, plot the result, and choose a value.
4. Engineer a feature, evaluate its value empirically, and explain why it helps one model and not another.
5. Know about: Some features are safe to build before running the pipeline.  Some should only be learned inside it. 

<a id="setup"></a>

## Setup: the data and pipeline from Checkpoint 1 Guidance

These cells repeat *Checkpoint 1 Guidance*'s Section 5, so this notebook runs on its own: the Adult census data, the same preprocessing (impute, scale, and one-hot encode), and the same 5 folds. Reread that section if any of it is unfamiliar. The data set has 48,842 rows, and 23.9% of people earn above \$50K.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
from sklearn.datasets import fetch_openml

adult = fetch_openml('adult', version=2, as_frame=True).frame   # download the Adult dataset from OpenML

columns = ['age', 'workclass', 'education-num', 'marital-status', 'occupation',
           'relationship', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
X = adult[columns]
y = (adult['class'] == '>50K').astype(int)      # 1 = income above $50K, 0 = not
X.head()

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_columns = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_columns = ['workclass', 'marital-status', 'occupation', 'relationship', 'native-country']

numeric_pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
categorical_pipeline = make_pipeline(SimpleImputer(strategy='constant', fill_value='missing'),
                                     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
preprocess = make_column_transformer((numeric_pipeline, numeric_columns),
                                     (categorical_pipeline, categorical_columns),
                                     verbose_feature_names_out=False)   # keeps the original column names

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(n_splits=5, shuffle=True, random_state=0)
floor = cross_val_score(DummyClassifier(), X, y, cv=folds)
print(f'Majority-class floor: {floor.mean():.1%}')

<a id="section-1"></a>

## 1 · A menu of model families

Each entry below is a complete pipeline: the preprocessing, followed by one model. To use one in your project, copy it and use your own `preprocess` from Checkpoint 1. Nothing else has to change.

All five entries use the same `preprocess`. Trees don't need scaling, because a split only asks which side of a threshold a value falls on, but scaling doesn't hurt them either. So one preprocessing step serves every model.

Each entry lists the one or two settings worth changing first. Every setting not listed keeps scikit-learn's default. `random_state=0` fixes the randomness in models that have it (a forest's bootstrap samples, a network's starting weights); in your project, use `random_state=SEED`.

### Logistic regression

You met it in Chapter 4 and used it in Checkpoint 1.

- **`C`** sets the strength of a penalty on the coefficients. By default, scikit-learn's logistic regression adds a ridge (L2) penalty, and `C` is the *inverse* of its strength: a smaller `C` means a stronger penalty and smaller coefficients, like a larger $\lambda$ in *Ridge and Lasso: Charging for Big Coefficients*. The default is `C=1`. Try values that differ by factors of 10, such as 0.01, 0.1, 1, and 10.
- **Needs scaled numeric columns**, because the penalty treats every coefficient alike, and a coefficient's size depends on its column's units.
- `max_iter=1000` gives the solver enough iterations to converge. With the default of 100, you may see a `ConvergenceWarning`.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_pipeline = make_pipeline(preprocess, LogisticRegression(C=1, max_iter=1000))

### K-nearest neighbors (KNN)

You met it in Chapters 2 and 4, and some of you used it in Checkpoint 1.

- **`n_neighbors`** is $K$. A small $K$ gives a flexible, high-variance model; a large $K$ gives a smoother one. The default is 5. Try values from about 5 to 100.
- **Needs scaled numeric columns**, because it measures distances, and an unscaled column with a large range would dominate them.
- It does no work at fit time and all its work at prediction time, comparing each new row with every training row, so it slows down as the data grows.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = make_pipeline(preprocess, KNeighborsClassifier(n_neighbors=25))

### Random forest

From *Ensembles: Bagging, Random Forests, and Boosting*, Section 3.

**A refresher.** A single deep tree is high variance: small changes in the training data give a very different tree. A random forest grows many deep trees, each on a bootstrap sample of the training rows, and averages their predictions (for classification, it averages their predicted probabilities). At every split, a tree considers only a random subset of $m$ of the $p$ features, which makes the trees less alike, so their errors cancel more when averaged.

- **`min_samples_leaf`** is a limit from *Trees and Regularization*: every leaf must hold at least this many training rows. The default, 1, lets every tree grow until its leaves are pure. On this data, that default is the forest's worst setting; [Section 3](#section-3) tunes it.
- **`max_features`** is $m$, the number of features considered at each split. The default, `'sqrt'`, is ISL's suggestion, $m \approx \sqrt{p}$.
- **`n_estimators`** is $B$, the number of trees. More trees never cause overfitting; they only take longer. 200 is plenty here.
- `n_jobs=-1` builds the trees in parallel on all of your computer's processor cores, which makes the forest much faster.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_pipeline = make_pipeline(preprocess,
                                RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0))

### Gradient boosting

From *Ensembles: Bagging, Random Forests, and Boosting*, Section 4.

**A refresher.** A random forest grows deep trees independently and averages them. Boosting does the opposite: it builds **small** trees one after another, and each new tree is fit to the errors the model *so far* still makes (its residuals). A shrunken copy of each tree is added to the model, so the model improves in many small corrections. The trees are regression trees even for classification: they add up a numeric score, the log-odds of class 1, and the sigmoid turns that score into a probability, as in logistic regression.

`HistGradientBoostingClassifier` is scikit-learn's fast implementation of this model.

- **`learning_rate`** is $\eta$, the shrinkage applied to each tree. The default is 0.1.
- **`max_iter`** is $B$, the number of trees. (The name is misleading: each iteration adds one tree.) The default is 100. A smaller $\eta$ needs a larger $B$, so tune the two together.
- **`max_leaf_nodes`** limits the size of each tree. The default is 31.
- **Early stopping is on by default for large data sets.** With more than 10,000 training rows, it sets aside 10% of them and stops adding trees once the score on those rows stops improving, so `max_iter` becomes an upper limit. Adult is large enough for that. Spaceship Titanic, with 6,519 training rows, isn't, so there it builds all `max_iter` trees.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosting_pipeline = make_pipeline(preprocess, HistGradientBoostingClassifier(random_state=0))

### Neural network (multilayer perceptron)

From Lab 7.

- **`hidden_layer_sizes`** is a tuple with one entry per hidden layer: `(20,)` is one hidden layer of 20 units, and `(20, 10)` is two hidden layers. The default is `(100,)`.
- **`alpha`** is the strength of an L2 penalty on the weights, called **weight decay**: the same idea as ridge regression, applied to a network. A larger `alpha` means smaller weights and a smoother model. The default is 0.0001. Try values that differ by factors of 10.
- **`early_stopping=True`** sets aside 10% of the training rows and stops training once accuracy on those rows hasn't improved by more than `tol` for `n_iter_no_change` epochs in a row: Lab 7's stopping rule, applied to held-out accuracy instead of the training loss. It's off by default.
- **Needs scaled numeric columns**, for the same reason as Lab 7: large inputs push the units into the flat parts of their activation functions, where gradients are tiny.
- Its default activation is `'relu'`, not the `'logistic'` units of Lab 7. Either works.

In [ ]:
from sklearn.neural_network import MLPClassifier

network_pipeline = make_pipeline(preprocess,
                                 MLPClassifier(hidden_layer_sizes=(20,), early_stopping=True,
                                               random_state=0))

<a id="section-2"></a>

## 2 · Comparing families on the same folds

Every model below is scored on the **same folds** (`folds`), so the comparison is meaningful: each model is validated on exactly the same rows.

**Code worth reading.** `compare` cross-validates each model in a dictionary and returns one row per model: the mean and standard deviation of its 5 fold accuracies, each fold's accuracy, all in percent, and how long it took. `final.ipynb` hands you this same function.

In [ ]:
def compare(models, X, y, folds):
    # Cross-validate each model on the same folds; return one row per model.
    rows = []
    for name, model in models.items():
        start = time.perf_counter()
        scores = cross_val_score(model, X, y, cv=folds)
        row = {'model': name,
               'CV accuracy (%)': 100 * scores.mean(),
               'std (%)': 100 * scores.std()}
        for k, score in enumerate(scores, start=1):
            row[f'fold {k}'] = 100 * score       # the same 5 folds for every model
        row['seconds'] = time.perf_counter() - start
        rows.append(row)
    return pd.DataFrame(rows).set_index('model').round(2)

Which family do you expect to do best on this data? Which do you expect to be slowest?

In [ ]:
models = {'logistic regression': logistic_pipeline,
          'KNN (K = 25)': knn_pipeline,
          'random forest': forest_pipeline,
          'gradient boosting': boosting_pipeline,
          'neural network (20 units)': network_pipeline}
family_table = compare(models, X, y, folds)
family_table

### How to tell whether a performance difference is significant 

Every CV accuracy is an estimate: a different split into folds would give slightly different numbers. Before you conclude that one model beats another, ask three questions.

1. **How big is the gap, compared with the fold-to-fold spread?** Define "the gap", as the difference between two models' CV accuracy (%) values. If the gap is several times larger than their respective std (%) values, it's very unlikely to be an accident of which rows landed in which fold.
2. **Is it consistent across folds?** Every model is scored on the same folds, and a fold that's hard for one model tends to be hard for all of them, so the models' scores rise and fall together. Compare the `fold` columns of two rows, fold by fold. A model that wins in every fold is more convincing than one that wins only on average, and a consistent win can be real even when the gap is smaller than the spread.
3. **Is it big enough to matter?** Convert the gap into a passenger count: how many more test passengers the better model would classify correctly. This notebook uses the Adult data, not Spaceship Titanic, so its gaps don't literally count passengers. But your project is scored on Spaceship Titanic's test set of 2,174 rows, so each time this notebook finds a gap, it also says what a gap that size would mean if you saw it in your project. There, a gap of 1 percentage point would be a passenger count of about 22 more correct predictions, and a gap of 0.1 points a passenger count of about 2.

If you decide some performance gap is not meaningful, evaluate the models on some other criteria, such as by favoring the simpler or faster one.

**Ask:** Use the three questions on the table above. Is gradient boosting's lead over the neural network real? Is the neural network's lead over logistic regression real?

<details>
<summary>Answer</summary>

- **Gradient boosting over the neural network: real, and big enough to matter.** The gap is 1.6 points (87.23% against 85.63%), about five times the spread of about 0.3 points. Boosting wins in every fold, by 1.21 to 1.81 points. If you saw a 1.6-point gap in your project, it would be a passenger count of about 35 on Spaceship Titanic's test set.
- **The neural network over logistic regression: probably real, but small.** The gap is 0.47 points (85.63% against 85.16%), only a little more than the spread, so the first question alone isn't convincing. But the network wins in every fold, by 0.39 to 0.51 points, so the gap is consistent. In your project, a gap this size would be a passenger count of about 10. And the network takes longer to fit.

</details>

Two more rows are worth a look:

- **Logistic regression and KNN are equally good.** The gap is 0.07 points (85.16% against 85.09%), and the two trade wins from fold to fold.
- **The random forest, at its default settings, is no better than logistic regression** (84.94%). Its trees grow until every leaf is pure, and averaging 200 of them reduces their variance but doesn't remove it. [Section 3](#section-3) fixes that with one setting.

The `seconds` column varies from computer to computer, but the order usually holds. Spaceship Titanic has about 13% as many rows, so there every entry runs in a few seconds.

<a id="section-3"></a>

## 3 · Tuning one setting

A model's default settings are a starting point, not a recommendation for your data. To tune a setting:

1. Pick one setting from the menu, for a model worth improving.
2. Choose 5–7 values spread widely, often by factors of 2 to 10, since you don't yet know where the good values are.
3. Cross-validate one pipeline per value, on the same folds, and plot the mean ± standard deviation against the value.
4. Choose a value with the three questions from [Section 2](#section-2): among the values whose gap from the best is too small to matter, prefer the simplest model.

Here, the setting is the random forest's `min_samples_leaf`. A larger value forces smaller trees, so a larger value means a simpler forest.

**Code worth reading.** The dictionary comprehension builds one pipeline per value, and `compare` scores them all. This cell takes about 30 seconds.

In [ ]:
leaf_sizes = [1, 2, 5, 10, 20, 50, 100]
forests = {leaf: make_pipeline(preprocess,
                               RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf,
                                                      n_jobs=-1, random_state=0))
           for leaf in leaf_sizes}
leaf_table = compare(forests, X, y, folds)
leaf_table

**Code worth reading.** `plt.errorbar` draws each mean with a bar one standard deviation above and below it. The values grow by roughly a factor of 2 each, so a logarithmic x-axis spaces them evenly.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(leaf_table.index, leaf_table['CV accuracy (%)'], yerr=leaf_table['std (%)'],
            marker='o', capsize=4)
ax.set_xscale('log')
ax.set_xticks(leaf_sizes, labels=leaf_sizes)      # label the values we tried, not powers of 10
ax.set_title('Random forest: CV accuracy against min_samples_leaf')
ax.set_xlabel('min_samples_leaf (log scale)')
ax.set_ylabel('CV accuracy (%), mean ± std')
plt.show()

Which value would you choose? Use the three questions from Section 2 on the two best values, 2 and 5.

<details>
<summary>Answer</summary>

`min_samples_leaf=1` is clearly worst (84.94%). The best is 2 (86.58%), and 5 (86.45%) is 0.13 points behind it.

- **Spread:** 0.13 points is well inside the spread of about 0.35 points.
- **Consistency:** 2 beats 5 in every fold, by 0.01 to 0.25 points, so the gap is probably real.
- **Size:** if you saw a 0.13-point gap in your project, it would be a passenger count of about 3 on Spaceship Titanic's test set. And 2's score is slightly optimistic, because it's the best of seven values chosen on these same folds (see *Why the winning value's CV score is a little optimistic*, below).

So 2 is probably a little better, but not by enough to matter. I would prefer the simpler model: **5**. Past 5, accuracy falls slowly as the trees get smaller (85.52% at 100), because they become too biased to capture important/predictive patterns. My intuition tells me that 2 samples per leaf is not worth the small gain in performance over 5. I suspect the performance cost of choosing 5 is worth the gain in regularization, but this is truly a judgement call. There are different schools of thought and a whole science to making decisions like this, which are beyond the scope of this course. Choosing 2 is definitely defensible.

</details>

Tuning raised the forest from 84.94% to 86.45%, but it still trails gradient boosting by about 0.8 points. In your project, a gap that size would be a passenger count of about 17. Tuning one model doesn't always change the ranking.

<details>
<summary>Optional: why the winning value's CV score is a little optimistic</summary>

The same 5 folds did two jobs in this section. We used them to *choose* `min_samples_leaf`, by picking the value with the highest mean accuracy over those folds, and then we report that same mean as the chosen forest's accuracy. But every value's mean is partly chance: it depends on which rows happened to land in which fold. Picking the highest of seven means tends to pick a value that was a little lucky, so the winner's mean is likely a bit higher than the accuracy it will get on rows it has never seen, like Spaceship Titanic's test set. The more values you try, the more chances one of them has to be lucky, and the larger this bias. It's one reason a tuned model can score a little lower on the leaderboard than in CV.

</details>

<a id="section-4"></a>

## 4 · Engineering features

A **feature** is a column the model uses. **Engineering** a feature means building a new column from the ones you have. Good engineered features provide the model with information/signal it can leverage effectively but struggles to build for itself. New features usually come from one of four places:

<details>
<summary>Optional: other meanings of "feature"</summary>

In this course, as in most work with tables of data, a feature is an input column: one coordinate of a point in feature space. Other areas of machine learning also use the word "feature" to mean patterns a model *learns* and represents internally, which nobody chose as a column.

A clear example is **mechanistic interpretability**, the study of how a trained network computes its outputs, often applied to transformer language models. A transformer keeps a vector for each token of text. Each layer reads that vector and **adds** its output to it, instead of replacing it. This running sum is called the **residual stream**. It's a hidden representation, like h-space in *From One Neuron to a Network*, Section 4, though not quite the same thing: there, each layer has its own space, while every layer of a transformer reads from and writes to the same one.

In that setting, a feature is a **direction** in the residual stream's space. 

None of this is needed for the final. In this notebook, "feature" always means a column.

</details>


| Source | Example |
| --- | --- |
| A column that packs several facts into one value | Splitting `Cabin` into deck, number, and side (Checkpoint 1) |
| A special value that means something on its own | A 0/1 column for "had any capital gain at all" |
| Several columns that belong together | Net capital: capital gain minus capital loss |
| Information in text, or in related rows | A title (Mr., Mrs., Master) inside a name; see [below](#titles) |

A note on that last row: Titles, such as Mr. Mrs, etc (also called *honorifics*) carry information. In 1912, "Master" was a standard formal title for a boy too young to be called "Mr.", so a passenger titled "Master" was a child. If your goal is to predict who survived the disaster, age was a very important feature. Many of the age values were missing, so it was useful to engineer a new feature based on title.

Here are two `Adult` dataset features, built and tested in three steps: one that helps a model, and one that doesn't.

The first is **age squared**, a new column holding each person's age times itself. The second is **net capital**: capital gain minus capital loss.

**Code worth reading.** Step 1 adds both new columns to a copy of `X`. Each uses only its row's own values, so it's safe to build before cross-validation. In your project, add the same columns to `X_test` too, or the model won't find them when you predict.

In [ ]:
X_more_features = X.copy()
X_more_features['age-squared'] = X['age'] ** 2
X_more_features['net-capital'] = X['capital-gain'] - X['capital-loss']
X_more_features[['age', 'age-squared', 'capital-gain', 'capital-loss', 'net-capital']].describe().round(1)

**Code worth reading.** Step 2 builds one new preprocessing step per feature. Each is the same as `preprocess`, except that the numeric list has one more name. Don't skip that name: a column that isn't in either list is silently dropped, so the model would never see your feature. A column transformer uses only the columns it's given, so the original `preprocess` simply ignores both new columns. That's why the pipelines with and without each feature can all run on the same `X_more_features`.

Step 3 compares the versions with `compare`, for logistic regression and for gradient boosting.

**Ask:** Before you run the next cell, predict: will `age-squared` help logistic regression? Will it help gradient boosting? What about our newly engineered, `net-capital` feature?

In [ ]:
age_squared_preprocess = make_column_transformer(
    (numeric_pipeline, numeric_columns + ['age-squared']),    # the one change: a new numeric column
    (categorical_pipeline, categorical_columns),
    verbose_feature_names_out=False)
net_capital_preprocess = make_column_transformer(
    (numeric_pipeline, numeric_columns + ['net-capital']),
    (categorical_pipeline, categorical_columns),
    verbose_feature_names_out=False)

feature_models = {
    'logistic regression': logistic_pipeline,
    'logistic regression + age-squared':
        make_pipeline(age_squared_preprocess, LogisticRegression(max_iter=1000)),
    'logistic regression + net-capital':
        make_pipeline(net_capital_preprocess, LogisticRegression(max_iter=1000)),
    'gradient boosting': boosting_pipeline,
    'gradient boosting + age-squared':
        make_pipeline(age_squared_preprocess, HistGradientBoostingClassifier(random_state=0)),
    'gradient boosting + net-capital':
        make_pipeline(net_capital_preprocess, HistGradientBoostingClassifier(random_state=0)),
}
feature_table = compare(feature_models, X_more_features, y, folds)
feature_table

**Age squared helps logistic regression.** Accuracy rises from 85.16% to 85.54%. The gain of 0.38 points is a little more than the spread, it holds in every fold (by 0.23 to 0.63 points), and if you saw a gain this size in your project, it would be a passenger count of about 8 on Spaceship Titanic's test set. Small, but real.

**Why does squaring a column help a linear model?** A full explanation is beyond the scope of this notebook. In brief, logistic regression gives `age` a single coefficient, so on its own `age` can only push the log-odds steadily up or steadily down. Adding `age` squared effectively lets the model fit a quadratic instead. To read more, search the web for "polynomial features logistic regression" or "feature transformations for linear models". This example is included so you can see feature engineering make a difference. 

**Age squared doesn't change gradient boosting at all** (87.23% both ways). A tree can split `age` at any threshold, as many times as it needs, so it could already follow the rise and fall. Squaring `age` doesn't change the order of the values, so every split on `age` squared is a split on `age` the tree already had.

**Net capital doesn't help either model.** Logistic regression goes from 85.16% to 85.15%, and gradient boosting from 87.23% to 87.22%. In your project, gaps this small would be a passenger count of 0. 

- **Logistic regression can already build net capital.** Net capital is a linear combination of two columns the model already has, and a linear model can build any linear combination of its inputs.

Most features you try won't help, and a feature that didn't help, along with why, is worth reporting in your write-up.

The general lesson: **a feature only helps when it gives the model something it can't build for itself, but even then, it might not help.**

- **Linear models** like logistic regression benefit from flags (a 0/1 column for a special value), bins, transformations like `log1p`, and products of columns, because they can't bend or combine their inputs on their own.
- **Trees** benefit from sums, differences, and ratios of columns, because each split looks at only one column at a time. A total of several related columns can take a tree many splits to approximate.
- **Every model** benefits from information that isn't in any column yet, such as the pieces of a packed string, or something learned from related rows.

**You don't need tricks like this for the final.** Squaring, logs, and other math on a numeric column are optional. A feature built from information the model can't otherwise use, such as the pieces of a string or something shared by related rows, fully meets the feature requirement, and on Spaceship Titanic it's usually the more useful kind. The [titles example below](#titles) shows how.

<a id="organized"></a>

### Optional: keeping feature code organized

This subsection isn't required. The final asks for one engineered feature, and the three steps above are all you need for it. If you try several features, though, this organization saves work and prevents two common mistakes.

With several features, it's easy for `X` and `X_test` to end up with different columns, or for a new column to never reach the model. Two functions and one list prevent both.

**Code worth reading.** A helper function, like `add_features` below, builds every new column, so applying it to both `X` and `X_test` guarantees they get the same ones. It returns a new DataFrame instead of mutating its argument. `new_numeric_columns` lists the numeric columns it creates, by name.

**Only some features belong in a helper like this.** `add_features` runs once, on all of `X` and all of `X_test`, before cross-validation, so it has to be safe to run there. Two kinds of feature aren't:

- **Anything built from the label** (`Transported`, or `class` here), such as a category's transported rate. That leaks the answer into the features, and the test rows have no labels to build it from anyway.
- **Anything that learns a number from the training rows**, such as a median to fill missing values or a mean to scale by. These belong inside the pipeline, so they're refit on each fold's training rows only.

What's left is features that use a row's own values, like the two here, or counts across rows that don't use the label. [The rule for where a feature goes](#feature-rule), at the end of this section, spells this out with examples.

<details>
<summary>Why bother? Why not build every feature inside cross-validation, to be safe?</summary>

For a feature that uses only a row's own values, there's nothing to be safe from. Squaring a row's age gives the same answer whether the other rows are in the training fold or not, so building it inside cross-validation would give exactly the same columns. Building it once, up front, has three advantages:

- **Simpler code.** 
- **Less repeated work.** Inside the pipeline, every feature is rebuilt for every fold of every model you compare or tune. For a cheap feature like age squared, that costs little; for a slow one, it adds up.
- **Sometimes you have to build it once** One kind of feature actually comes out *wrong* inside cross-validation: a count across rows, like the size of a travel group. Inside cross-validation, each count would see only some of the rows, so a group split across the training and validation rows would look smaller than it is.

</details>


In [ ]:
def add_features(df):
    df = df.copy()                  # a new DataFrame; the caller's df isn't mutated
    df['age-squared'] = df['age'] ** 2
    df['net-capital'] = df['capital-gain'] - df['capital-loss']
    # ... one line per new feature
    return df

new_numeric_columns = ['age-squared', 'net-capital']    # must match the names add_features creates

X_more_features = add_features(X)
# In your project, also: X_test_more_features = add_features(X_test)

**Code worth reading.** `make_preprocess` builds the same preprocessing as `preprocess` (defined way at the top in Setup) from the column lists it's given. 

In [ ]:
def make_preprocess(numeric_columns, categorical_columns):
    numeric_pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
    categorical_pipeline = make_pipeline(SimpleImputer(strategy='constant', fill_value='missing'),
                                         OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    return make_column_transformer((numeric_pipeline, numeric_columns),
                                   (categorical_pipeline, categorical_columns),
                                   verbose_feature_names_out=False)

**Using them together.** Below, we'll call the `make_pipeline` function with different arguments, so we can compare the resulting models. The "without" version gets the original feature lists, and the "with" version gets the original feature lists plus `new_numeric_columns` (our newly engineered features). Both run on the same `X_more_features`, but if the feature names are not passed into `make_pipeline` then they're ignored by the model. Here the "with" version uses both new features at once. It scores 85.56%, about the same as age squared alone (85.54%), because net capital adds nothing useful to `LogisticRegression`.

In [ ]:
organized_models = {
    'logistic regression':
        make_pipeline(make_preprocess(numeric_columns, categorical_columns),
                      LogisticRegression(max_iter=1000)),
    'logistic regression + new features':
        make_pipeline(make_preprocess(numeric_columns + new_numeric_columns, categorical_columns),
                      LogisticRegression(max_iter=1000)),
}
organized_table = compare(organized_models, X_more_features, y, folds)
organized_table

**Check that your feature reaches the model.** The names in `new_numeric_columns` must match the names `add_features` creates. A misspelled name raises an error when the pipeline is fit, but a feature you forget to list is **silently dropped**: the column is in the data, and the model never sees it. To see which columns a preprocessing step actually passes on, fit it and call `get_feature_names_out()`.

In [ ]:
with_columns = make_preprocess(numeric_columns + new_numeric_columns, categorical_columns)
used = with_columns.fit(X_more_features).get_feature_names_out()
print(f"'with' version: {len(used)} columns; includes the new features: {set(new_numeric_columns) <= set(used)}")

without_columns = make_preprocess(numeric_columns, categorical_columns)
used = without_columns.fit(X_more_features).get_feature_names_out()
print(f"'without' version: {len(used)} columns; includes the new features: {set(new_numeric_columns) <= set(used)}")

**For the submission**, fit the version you chose on every training row, then predict for the test rows.

In [ ]:
final_model = organized_models['logistic regression + new features']
final_model.fit(X_more_features, y)        # every training row
# In your project: predictions = final_model.predict(X_test_more_features)

With this setup, adding a feature takes two edits: one line in `add_features`, and its name in `new_numeric_columns`. A new categorical feature works the same way, with a second list added to `categorical_columns`.

<a id="titles"></a>

### Titles in names: a safe way and a leaky way

Here's an example of feature engineering using honorifics (titles) from passenger names:

In [ ]:
passengers = pd.DataFrame({ # Example data for Titanic passengers 
    'Name': ['Ashby, Mr. Thomas', 'Ashby, Mrs. Clara', 'Ashby, Master. Henry',
             'Pell, Miss. Ada', 'Pell, Mr. George', 'Okafor, Mr. Daniel',
             'Okafor, Miss. Grace', 'Rossi, Mrs. Maria', 'Rossi, Master. Luca',
             'Vane, Countess. Eleanor'],
    'Age': [34, 31, np.nan, 19, np.nan, 45, 22, 36, 4, 38],
    'Survived': [0, 1, 1, 1, 0, 1, 0, 0, 0, 1],
})
passengers

**Step 1: extract the title. This is safe.** It uses only the row's own text, like splitting `Cabin` in Checkpoint 1, so you can do it once, to all of `X` and `X_test`, before cross-validation splits the rows into folds.

**Code worth reading.** `.str.extract` takes a regular expression and returns the part inside the parentheses. `,\s*([^.]+)\.` means: a comma, any spaces, then capture everything up to the next period.

In [ ]:
passengers['Title'] = passengers['Name'].str.extract(r',\s*([^.]+)\.', expand=False)
passengers[['Name', 'Title']]

**Step 2, the WRONG ("leaky") way: replace each title with its survival rate.** This is called **target encoding**, and done this way, before cross-validation, it leaks information about the label/answers across the training vs validation boundary.

**Extreme Example:** Look at the Countess's row in the output below. Why is her `title survival rate` a problem?

In [ ]:
leaky = passengers.copy()
leaky['title survival rate'] = leaky.groupby('Title')['Survived'].transform('mean')   # uses the labels!
leaky[['Title', 'Survived', 'title survival rate']]

<details>
<summary>Answer</summary>

She's the only Countess, so her title's survival rate is computed from her label alone: it *is* her outcome, 1.0. When her row lands in a validation fold, the model is validated on a feature that already contains the answer. Every other rate includes its own row's label too, just diluted. Cross-validation looks better than the leaderboard will, because the test rows have no labels to build this column from.

</details>

A milder version of the same mistake is filling each missing age with the median age for that title, computed on all of the training data before cross-validation. That doesn't use the class label (answer), so it leaks much less: the validation rows' ages help compute the fill values, which barely change when one fold is removed. It's the same kind of small leak as fitting a scaler before cross-validation.

**Step 3, the CORRECT way: keep the title as a column, and let the pipeline handle the rest.** Treat `Title` as one more categorical column, and leave the plain median imputer for `Age` inside the pipeline. `add_indicator=True` also adds a 0/1 column that says whether the age was missing. (This example leaves out the scaler so its output is easy to read; a linear model would still need one.) A tree ensemble can then learn the combination on its own: a passenger with title, "Master", whose age is missing is a child.

In [ ]:
titanic_preprocess = make_column_transformer(
    (SimpleImputer(strategy='median', add_indicator=True), ['Age']),   # A flag the model can use to identify missing Age values
    (categorical_pipeline, ['Title']), # Encode the Title column using the categorical pipeline (assumes Title column exists)
    verbose_feature_names_out=False)
pd.DataFrame(titanic_preprocess.fit_transform(passengers),
             columns=titanic_preprocess.get_feature_names_out()).round(2)

Henry's missing age is filled with 32.5, the median of everyone with a known age, which is wrong for a boy. But his row also has `missingindicator_Age = 1` and `Title_Master = 1`, and together those two columns allow models to learn the correct pattern, in this case, basically ignore the imputed age and rely on the title.

<a id="feature-rule"></a>

### The rule for where a feature goes

| The feature... | Where it goes | Examples |
| --- | --- | --- |
| uses only a row's own values, or other columns of `X` and `X_test` | **Before cross-validation**: once, to all of `X` and `X_test`, outside the pipeline | Splitting a string; a 0/1 flag; a sum of columns; a title; how many passengers share a travel group |
| learns a number from the training rows | **Inside the pipeline**, so it's refit on each fold's training rows only | A median or mean to fill gaps; a scaler; one-hot categories |
| uses the target (`Transported`) | **Nowhere** in this project | A category's transported rate |

<details>
<summary>Optional: why does one-hot encoding belong inside the pipeline?</summary>

One-hot encoding adds no information; it's another way to write a category. But `OneHotEncoder` does learn something when it's fit: the list of categories, one column each. Fit before cross-validation, that list would include categories that appear only in a fold's validation rows. That barely matters, since the model sees such a column as all zeros during training and learns nothing from it. Two reasons to keep the encoder in the pipeline anyway:

- **Categories in validation but not in test:** `X_test` may contain a category the training rows never had. Fit inside each fold, so set the encoder to meet unseen categories in cross-validation using `handle_unknown='ignore'`, which encodes them as all zeros because if they weren't used to train the model in CV, they shouldn't be used for predictions.
- **`X_test` gets exactly the training columns,** in the same order, because the fitted pipeline encodes it. Encoding each file separately by hand, as with `pd.get_dummies`, is a common way to end up with mismatched columns.

</details>

Counts computed across `X` and `X_test` together, like the size of a travel group, are fine. They don't use the label, and you'll have the test rows' features when you predict.

**Your data has its own version of the title idea.** Spaceship Titanic's names don't carry titles, but related rows still carry information. Look at what passengers in the same travel group have in common.

<a id="section-5"></a>

## 5 · Optional: the wrong way to cross-validate

This section isn't graded. It shows *why* every step that learns from the data belongs inside the pipeline, with an example where the mistake is dramatic.

Here's a data set with nothing to learn: 50 samples with 5,000 features, every value random noise, and labels assigned by coin flip.

In [ ]:
rng = np.random.default_rng(465)
X_noise = rng.normal(size=(50, 5000))     # 50 samples, 5,000 features of pure noise
y_noise = rng.integers(0, 2, 50)          # coin-flip labels

**Ask:** These labels are coin flips. What error rate should an honest method get?

<details>
<summary>Answer</summary>

About 50%. There's nothing to learn, so any honest estimate should be close to guessing.

</details>

**Code worth reading: the wrong way.** `SelectKBest(f_classif, k=100)` keeps the 100 features most related to the labels. Here it looks at all 50 samples, and then we run 5-fold cross-validation of KNN with $K = 1$ on those 100 features.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

X_selected = SelectKBest(f_classif, k=100).fit_transform(X_noise, y_noise)    # looks at every label
scores = cross_val_score(KNeighborsClassifier(n_neighbors=1), X_selected, y_noise, cv=5)
print(f'Wrong way: cross-validation error = {1 - scores.mean():.0%}')

**Code worth reading: the right way.** The pipeline bundles the steps, so cross-validation repeats all of them in every fold. Feature selection now happens inside each fold, using only that fold's training rows.

In [ ]:
selection_pipeline = make_pipeline(SelectKBest(f_classif, k=100), KNeighborsClassifier(n_neighbors=1))
scores = cross_val_score(selection_pipeline, X_noise, y_noise, cv=5)
print(f'Right way: cross-validation error = {1 - scores.mean():.0%}')

In the wrong version, the feature selection looked at every label, including the labels of rows that later served as validation rows, so those rows weren't really new. Out of 5,000 noise features, some match the coin flips by chance, and choosing them with every label in view builds that chance match into the "validation" score. That's the same mistake as the title survival rate in [Section 4](#titles): a feature built from the labels of the rows it's validated on.

This mistake has appeared in published research, especially in genomics, where data sets have thousands of features and few samples. Imputers and scalers leak much less than feature selection does, but the habit is the same, and a pipeline is worth the effort.

<a id="summary"></a>

## Summary

1. **A menu, not a search:** each family is one line inside the same pipeline. Copy an entry, keep your own `preprocess`, and change one or two settings.
2. **Compare fairly:** the same folds for every model, in one table of mean ± standard deviation and the fold-by-fold scores. Before you believe a difference, ask how big it is compared with the spread, whether it holds in every fold, and whether its passenger count is big enough to matter. When it isn't, keep the simpler or faster model.
3. **Tune one setting** by cross-validating 5–7 widely spread values and plotting the results. Among the values whose gap from the best is too small to matter, prefer the simplest model.
4. **A feature helps when it gives the model something it can't build for itself:** flags and transformations for linear models, combinations of columns for trees, and new information for every model. Test it with and without, on the same model and folds.
5. **Where a feature goes:** built from the row (or from `X` and `X_test` without the label) → before cross-validation; learned from the training rows → inside the pipeline; built from the target → nowhere.